# RL-RC-DoT → VVenC: Mini Pilot (256 transitions) — adapter/3

**Mini BDD100K = 16 Train / 4 Validation / 30 Test**, chọn từ `dataset_lock.json` gốc theo seed cố định và phân tầng theo mật độ đối tượng. File lock không thay đổi. Train dùng cửa sổ 256 frame chọn từ các vị trí khác nhau, Val và Test cố định tại đầu video.

Pilot 256 transitions chỉ kiểm tra native preflight, PPO và Validation ngắn trên 2 video × 64 frame.

Bật Kaggle T4 × 2, Internet và đính kèm các video MOV BDD100K. Notebook kiểm tra đủ Mini Train/Val **trước khi build**. Bắt đầu MODE=new. Không resume adapter/2 hay checkpoint Full65/7.


In [ ]:
import time,sys,subprocess,json,hashlib,zipfile
from pathlib import Path
if 'RCDOT_SESSION_START_UTC' not in globals():
    RCDOT_SESSION_START_UTC=time.time()

SOURCE_COMMIT='7086a402b16dcacc46eed59339d101e3a9c73096'
PHASE='pilot'                 # pilot=256; trial=4096; full=training từng đợt
MODE='new'                    # Chỉ resume cùng Mini/config/source
SESSION_HOURS=10.0            # Giới hạn an toàn, không bắt chạy đủ 10 giờ; gồm setup
SEED=20261111
GPU_IDS='0,1'
WORKERS=2
PPO_LEARNING_RATE=3e-5
PPO_EPOCHS=3
PPO_TARGET_KL=0.05
PPO_AUX_LOSS_COEF=0.1         # Ablation riêng: 0.0 (không dùng Aux); KHÔNG đổi trong một run
PPO_PROFILE={'learning_rate':PPO_LEARNING_RATE,'n_epochs':PPO_EPOCHS,'target_kl':PPO_TARGET_KL,'aux_loss_coef':PPO_AUX_LOSS_COEF}
DATASET_MODE='mini'
TRAIN_FRAME_LIMIT=256         # 256 frame mỗi episode
if PHASE=='pilot':
    ROLLOUT_STEPS=64
    ADDITIONAL_TIMESTEPS=256  # Số tương tác BỔ SUNG; SB3 làm tròn theo rollout
    VAL_VIDEOS,VAL_RATES,VAL_FRAMES=2,'50,120',64
elif PHASE=='trial':
    ROLLOUT_STEPS=256
    ADDITIONAL_TIMESTEPS=20480  # Mini Trial riêng; không resume Pilot
    VAL_VIDEOS,VAL_RATES,VAL_FRAMES=4,'50,120',256  # Mini Val4 x 256 frame
else:
    ROLLOUT_STEPS=2048
    ADDITIONAL_TIMESTEPS=20480 # Tăng dần bằng resume, không tự chạy 20 triệu
    VAL_VIDEOS,VAL_RATES,VAL_FRAMES=7,'50,80,120,160,200',64  # Reduced validation, không phải full clip
VAL_EVERY_UPDATES=2 if PHASE=='pilot' else (10 if PHASE=='trial' else 25)
TOTAL_TIMESTEPS=20_000_000
REPO=Path('/kaggle/working/vvenc_rl_source')
TOOLS=Path('/kaggle/working/rl_rc_dot_tools')
RUN=Path('/kaggle/working/rcdot_mini_'+PHASE+'_adapter3_v1')
assert PHASE in {'pilot','trial','full'} and MODE in {'new','resume'}
assert 0 <= PPO_AUX_LOSS_COEF <= 1
assert 0<SESSION_HOURS<=11.75
print('CONFIG',PHASE,MODE,'WORKERS',WORKERS,'ADDITIONAL_INTERACTIONS',ADDITIONAL_TIMESTEPS)
print('SOURCE_COMMIT',SOURCE_COMMIT,'DATASET_MODE',DATASET_MODE,'TRAIN_FRAMES',TRAIN_FRAME_LIMIT)
print('VALIDATION_EVERY_UPDATES',VAL_EVERY_UPDATES,'AUX_COEF',PPO_AUX_LOSS_COEF)
print('Full/adapter2 checkpoint INCOMPATIBLE with Mini adapter3; MODE=new first.')
print('Pilot/trial/full have distinct contracts. For continuation, preserve PHASE and PPO_PROFILE; set MODE=resume.')

import torch
print("GPU_COUNT",torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count()==2,"Bật GPU T4x2 trước khi setup"


In [ ]:
def live(command,cwd=None):
    print('$',' '.join(map(str,command)),flush=True)
    p=subprocess.Popen(list(map(str,command)),cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout:print(line,end='',flush=True)
        rc=p.wait()
    finally:
        if p.poll() is None:p.terminate();p.wait()
    if rc:raise RuntimeError('Command failed; inspect logs/exports in /kaggle/working')

if not (REPO/'.git').is_dir():
    live(['git','clone','--branch','feature/rl-rc-dot-vvc','--depth','1',
          'https://github.com/youngan2810/vvenc_rl.git',REPO])
assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=REPO,text=True).strip(),'Tracked source modified'
live(['git','fetch','--no-tags','--depth','1','origin',SOURCE_COMMIT],REPO)
live(['git','checkout','--detach',SOURCE_COMMIT],REPO)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==SOURCE_COMMIT
# Fail fast before installing/building the VVC toolchain.
live([sys.executable,'-c',"from scripts.rl_rc_dot.data import VideoCatalog; import json; c=VideoCatalog('/kaggle/input',mode='mini',seed=20261111); c.check_available(('train','val')); print('MINI_INPUT_PASS',json.dumps({k:c.selected[k] for k in ('train','val')}))"],REPO)
live([sys.executable,'-m','scripts.rl_rc_dot.setup_kaggle','--repo',REPO,'--source-commit',SOURCE_COMMIT,
      '--output',TOOLS,'--jobs','2','--install-system'],REPO)
toolchain=json.loads((TOOLS/'TOOLCHAIN.json').read_text())
live([sys.executable,'-c','import torch;print(torch.__version__,torch.version.cuda);print("GPU_COUNT",torch.cuda.device_count());assert torch.cuda.is_available() and torch.cuda.device_count()==2'])


In [ ]:
# Chọn đúng export mới; không tự hạ MODE=resume xuống new nếu thiếu Input.
RESUME_FROM=None
if MODE=='resume' and not (RUN/'progress.json').is_file():
    current_fingerprint=subprocess.check_output(
        [sys.executable,'-c','from scripts.rl_rc_dot.data import source_fingerprint;print(source_fingerprint())'],
        cwd=REPO,text=True).strip()
    def meta(path):
        if path.is_dir():
            return json.loads((path/'method.json').read_text()),json.loads((path/'progress.json').read_text())
        with zipfile.ZipFile(path) as z:
            if 'session_payload.bin' in z.namelist():
                with z.open('session_payload.bin') as payload,zipfile.ZipFile(payload) as inner:
                    return json.loads(inner.read('method.json')),json.loads(inner.read('progress.json'))
            return json.loads(z.read('method.json')),json.loads(z.read('progress.json'))
    candidates=[]
    inputs=Path('/kaggle/input')
    paths=list(inputs.rglob('session_payload.bin'))
    paths += [p for p in inputs.rglob('*.zip') if p.name.startswith('RL_RC_DOT_VVC_SESSION')]
    paths += [p.parent for p in inputs.rglob('export_manifest.json')]
    for path in paths:
        try:
            method,progress=meta(path)
            expected_val={'videos':VAL_VIDEOS,'rates':list(map(int,VAL_RATES.split(','))),'frames':VAL_FRAMES}
            if (method.get('schema')=='vvenc-rcdot-run/2' and method.get('source_fingerprint')==current_fingerprint
                and method.get('encoder_sha256')==toolchain['encoder_sha256'] and method.get('frames')==TRAIN_FRAME_LIMIT
                and method.get('workers')==WORKERS and method.get('rollout_steps')==ROLLOUT_STEPS
                and method.get('seed')==SEED and method.get('validation')==expected_val
                and method.get('ppo')==PPO_PROFILE and method.get('dataset_mode')==DATASET_MODE):
                candidates.append((progress['num_timesteps'],str(path)))
        except (ValueError,KeyError,FileNotFoundError,zipfile.BadZipFile):
            pass
    assert candidates,'Không có Input resume adapter/3 khớp source/profile/phase. Upload gói KAGGLE_SAFE mới; không dùng export v3/v4'
    candidates.sort(reverse=True)
    RESUME_FROM=Path(candidates[0][1])
    print('RESUME_SELECTED',RESUME_FROM,'OBSERVED_FRAMES',candidates[0][0])
elif MODE=='new':
    assert not RUN.exists() or not any(RUN.iterdir()),'Run đã tồn tại: đổi MODE=resume hoặc dùng Session mới'
    print('NEW_MINI_ADAPTER3_RUN — không nạp checkpoint cũ')
else:
    print('RESUME_EXISTING_WORKING_RUN',RUN)


In [ ]:
command=[sys.executable,'-u','-m','scripts.rl_rc_dot.session',
 '--input-root','/kaggle/input','--encoder',toolchain['encoder'],'--decoder',toolchain['decoder'],
 '--yolo-repo',toolchain['yolo_repo'],'--weights',toolchain['weights'],'--output',RUN,
 '--mode',MODE,'--dataset-mode',DATASET_MODE,'--workers',str(WORKERS),'--gpu-ids',GPU_IDS,'--encoder-threads','2',
 '--seed',str(SEED),'--frames',str(TRAIN_FRAME_LIMIT),'--total-timesteps',str(TOTAL_TIMESTEPS),
 '--additional-timesteps',str(ADDITIONAL_TIMESTEPS),'--rollout-steps',str(ROLLOUT_STEPS),
 '--ppo-learning-rate',str(PPO_LEARNING_RATE),'--ppo-epochs',str(PPO_EPOCHS),
 '--ppo-target-kl',str(PPO_TARGET_KL),'--ppo-aux-loss-coef',str(PPO_AUX_LOSS_COEF),
 '--validation-videos',str(VAL_VIDEOS),'--validation-rates',VAL_RATES,'--validation-frames',str(VAL_FRAMES),
 '--val-every-updates',str(VAL_EVERY_UPDATES),'--max-hours',str(SESSION_HOURS),
 '--session-start-utc',str(RCDOT_SESSION_START_UTC)]
if RESUME_FROM:command += ['--resume-from',str(RESUME_FROM)]
live(command,REPO)


In [ ]:
from IPython.display import display,FileLink
import shutil
progress=json.loads((RUN/'progress.json').read_text())
print('PROGRESS',json.dumps(progress,indent=2))
exports=sorted((RUN/'exports').glob('*_KAGGLE_SAFE.zip'),key=lambda p:p.stat().st_mtime)
assert exports,'Chưa có export; kiểm tra failure.json/logs'
safe=Path('/kaggle/working/RL_RC_DOT_VVC_LATEST_KAGGLE_SAFE.zip')
shutil.copyfile(exports[-1],safe)
print('FILE_READY',safe,'MB',round(safe.stat().st_size/1024**2,2))
print('Kaggle FileLink có thể trả về 404; nếu có lỗi, tải trong Notebook Output / Save Version.')
import os
os.chdir('/kaggle/working')
display(FileLink(safe.name))
print('Giữ lại toàn bộ export của từng phiên; gói SAFE này là Input resume của cùng phase/source/config')
print('Kiểm tra max_approx_kl_observed, stopped_by_target_kl, clip_fraction và actual bitrate; smoke/trial chưa chứng minh hội tụ hoặc VVenC gains')


In [ ]:
# 5. Xem chẩn đoán PPO sau smoke / trial (không cần mở /kaggle/working)
import json
from pathlib import Path
updates=RUN/'metrics'/'updates.jsonl'
if updates.exists():
    rows=[json.loads(s) for s in updates.read_text().splitlines() if s.strip()]
    cols=['completed_rollouts','trained_frames','approx_kl','clip_fraction','stopped_by_target_kl',
          'explained_variance','explained_variance_defined','return_rmse',
          'actor_gradient_norm','critic_gradient_norm','auxiliary_loss',
          'auxiliary_prediction_mean','auxiliary_target_mean']
    try:
        import pandas as pd
        display(pd.DataFrame(rows)[[c for c in cols if c in rows[0]]].round(4))
    except ImportError:
        for item in rows: print({key:round(item[key],4) if isinstance(item[key],float) else item[key] for key in cols if key in item})
else:
    print('Chưa có PPO update log.')


In [ ]:
# Mini diagnostics: small bundle without videos or model weights
from zipfile import ZipFile,ZIP_DEFLATED
diag=Path('/kaggle/working/RL_RC_DOT_MINI_'+PHASE.upper()+'_DIAGNOSTICS.zip')
sources=[RUN/'method.json',RUN/'dataset_selection.json',RUN/'session.json',RUN/'progress.json',
         RUN/'failure.json',RUN/'preflight/PREFLIGHT.json']
for sub in ('metrics','validations'):
    folder=RUN/sub
    if folder.exists():sources.extend(x for x in folder.rglob('*') if x.suffix in ('.json','.jsonl'))
with ZipFile(diag,'w',ZIP_DEFLATED) as z:
    for x in sources:
        if x.is_file():z.write(x,x.relative_to(RUN))
from IPython.display import FileLink,display
print('MINI_DIAGNOSTICS_READY',diag,diag.stat().st_size,'bytes')
display(FileLink(diag.name))
